# Qwen2.5-Coder-0.5B — Python FIM · W&B + Weave Tracked Training

This notebook runs the **full training + evaluation pipeline** with **Weights & Biases** and **Weave** tracking enabled.

Every run gets a **deterministic, human-readable ID** automatically built from your config:
```
qwen05b-lora-r16-e3-dsv1-20260810-a1
│        │    │   │   │    │        └─ attempt  (set in lora.yaml → wandb.attempt)
│        │    │   │   │    └────────── date      (today's UTC date, auto)
│        │    │   │   └─────────────── dataset   (set in lora.yaml → wandb.dataset_version)
│        │    │   └─────────────────── epochs    (set in lora.yaml → training.num_epochs)
│        │    └─────────────────────── LoRA rank (set in lora.yaml → lora.r)
│        └──────────────────────────── method    (always lora)
└───────────────────────────────────── model     (derived from model.name)
```

## Required Kaggle setup before running
| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 |
| Internet | **ON** |
| Dataset input | `fim-python-dataset` (your Kaggle dataset) |
| Secret: `HF_TOKEN` | Your Hugging Face **write** token |
| Secret: `WANDB_API_KEY` | Your W&B API key from https://wandb.ai/authorize |

## What gets logged to W&B
- ✅ All hyperparameters (from `configs/lora.yaml`)
- ✅ Training & validation loss curves (live, every `logging_steps`)
- ✅ Final adapter size + HF repo URL
- ✅ Eval metrics: Exact Match + Edit Similarity (Base vs LoRA-FT)
- ✅ Per-example results table (filterable in the W&B UI)
- ✅ Comparison bar chart + loss curve as W&B Images
- ✅ Weave traces for all model inference calls

In [ ]:
# ── Cell 1: Clone repository ─────────────────────────────────────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# ── Set these before running ─────────────────────────────────────────────────
BRANCH = "feat/stage-one"  # branch to clone (None → main)
COMMIT = None              # specific commit hash (None → latest of branch)

REPO_DIR = "/kaggle/working/repo"

if BRANCH is None:
    BRANCH = "main"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

if COMMIT:
    print(f"Cloning repository...")
    print(f"Branch : {BRANCH}")
    print(f"Commit : {COMMIT}")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", GITHUB_REPO, REPO_DIR],
        check=True,
    )
    os.chdir(REPO_DIR)
    subprocess.run(["git", "checkout", "--detach", COMMIT], check=True)
else:
    print(f"Cloning latest commit from branch: {BRANCH}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", GITHUB_REPO, REPO_DIR],
        check=True,
    )
    os.chdir(REPO_DIR)

current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
current_branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()

print("\n✓ Repository ready")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")

In [ ]:
# ── Cell 2: Install all dependencies (including wandb + weave) ───────────────
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
        "peft>=0.20.0",
        "trl>=0.17.0",
        "transformers>=5.0.0",
        "datasets>=3.0.0",
        "pyyaml>=6.0",
        "huggingface_hub>=0.30.0",
        "seaborn>=0.13.0",
        "pandas>=2.0.0",
        "wandb>=0.19.0",       # Weights & Biases
        "weave>=0.51.0",       # Weave tracing (from W&B)
    ],
    check=True,
)

print("✓ All dependencies installed (including wandb + weave)")

In [ ]:
# ── Cell 3: Load secrets (HF token + W&B API key) ────────────────────────────
# Both secrets must be added via: Kaggle → Settings → Secrets
#   HF_TOKEN      → your Hugging Face WRITE token (https://hf.co/settings/tokens)
#   WANDB_API_KEY → your W&B API key (https://wandb.ai/authorize)
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

print("✓ HF_TOKEN set")
print("✓ WANDB_API_KEY set")

In [ ]:
# ── Cell 4: Configure the run ─────────────────────────────────────────────────
# Edit these values before running. They become part of the run ID.
#
#   attempt         — increment each time you re-run with the same config
#   dataset_version — must match the Kaggle Dataset version attached in the sidebar
#
# The final run ID will look like:  qwen05b-lora-r16-e1-dsv1-20260810-a1

import yaml

CONFIG_PATH = f"{REPO_DIR}/configs/lora.yaml"

# ── Patch run-specific fields directly in the loaded config ──────────────────
ATTEMPT         = 1    # ← change this each re-run
DATASET_VERSION = "v1" # ← change this if you attach a new dataset version

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

cfg["wandb"]["attempt"]         = ATTEMPT
cfg["wandb"]["dataset_version"] = DATASET_VERSION

with open(CONFIG_PATH, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

# Preview the run ID that will be used
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from src.train_lora import load_config, build_run_id

cfg = load_config(CONFIG_PATH)
run_id = build_run_id(cfg)

print(f"✓ Config patched")
print(f"  Attempt         : {ATTEMPT}")
print(f"  Dataset version : {DATASET_VERSION}")
print(f"  Run ID          : {run_id}")

In [ ]:
# ── Cell 5: Copy dataset from Kaggle Dataset input ───────────────────────────
# In the Kaggle sidebar: Add Data → Your Datasets → fim-python-dataset
# Make sure the version matches DATASET_VERSION above.
import pathlib
import shutil

DATASET_INPUT = (
    "/kaggle/input/datasets/rudraprasadbhuyan/fim-python-dataset/data/fim_dataset.jsonl"
)
DATASET_DEST = f"{REPO_DIR}/data/fim_dataset.jsonl"

pathlib.Path(f"{REPO_DIR}/data").mkdir(parents=True, exist_ok=True)
shutil.copy(DATASET_INPUT, DATASET_DEST)

n = sum(1 for line in open(DATASET_DEST) if line.strip())
print(f"✓ Dataset copied: {n} FIM examples  (version: {DATASET_VERSION})")

In [ ]:
# ── Cell 6: Run LoRA training (W&B + Weave tracking active) ──────────────────
from src.train_lora import train

# W&B and Weave are initialised automatically inside train() because WANDB_API_KEY is set.
# The run URL will be printed after init.
wandb_run = train(config_path=CONFIG_PATH)

# train() returns the active wandb.Run so we can pass it to evaluate() below.
# If WANDB_API_KEY was not set, wandb_run will be None (training still works).

In [ ]:
# ── Cell 7: Evaluate — Base vs LoRA-FT (metrics logged to same W&B run) ──────
from src.evaluate import run_evaluation

summary = run_evaluation(
    base_model_name="Qwen/Qwen2.5-Coder-0.5B",
    adapter_path="/kaggle/working/checkpoints/lora_adapter",
    test_data_path="data/fim_dataset.jsonl",
    output_dir="/kaggle/working/results",
    max_samples=200,
    trainer_log_path="/kaggle/working/checkpoints/trainer_state.json",
    wandb_run=wandb_run,   # ← pass the run so eval metrics go into the same W&B run
)

print("\n📊  Final Summary")
print(summary)

In [ ]:
# ── Cell 8: Display plots inline + print W&B / Weave links ───────────────────
from IPython.display import Image, display

for plot in [
    "/kaggle/working/results/plots/loss_curve.png",
    "/kaggle/working/results/plots/comparison.png",
]:
    if __import__("pathlib").Path(plot).exists():
        print(f"\n{plot}")
        display(Image(filename=plot))

print("\n" + "=" * 60)
if wandb_run is not None:
    project = cfg.get("wandb", {}).get("project", "")
    print(f"✓ W&B run     → {wandb_run.url}")
    print(f"✓ W&B project → https://wandb.ai/{project}")
    print(f"✓ Weave traces→ https://weave.wandb.ai/{project}")
else:
    print("ℹ  W&B tracking was not active (WANDB_API_KEY not set).")
print("=" * 60)

## How to view your results on W&B

1. Go to **https://wandb.ai/rudraprasadbhuyan-ugie/qwen-coder-python-fim**
2. Click your run (named like `qwen05b-lora-r16-e1-dsv1-20260810-a1`)
3. Explore:
   - **Charts tab** — live training + validation loss, learning rate schedule
   - **Summary tab** — final metrics at a glance
   - **Config tab** — all hyperparameters (from `configs/lora.yaml`)
   - **Media tab** — loss curve + comparison bar chart images
   - **Tables tab** — per-example evaluation results (filterable)

## Comparing runs

When you change a hyperparameter (e.g. `lora.r: 32` or `training.num_epochs: 3`),
increment `wandb.attempt` in `configs/lora.yaml` before running again.
W&B will show both runs side by side in the project table.

## Weave traces

Go to **https://weave.wandb.ai/rudraprasadbhuyan-ugie/qwen-coder-python-fim** to see
a full trace of every model call made during the run.

## Resuming next week

```python
# Load your saved adapter from HF on top of a fresh base model:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base  = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-Coder-0.5B', device_map='auto')
model = PeftModel.from_pretrained(base, 'Rudra-G-23/qwen2.5-coder-0.5b-python-fim-lora')
# Then continue training or just evaluate.
```

The base model never changes. Only your adapter accumulates updates.